%md
# Exploratory Data Analysis (EDA)
## 10-Year Coronary Heart Disease Risk Prediction

### Objective

The objective of this project is to develop a machine-learning
model to predict whether a person is at risk of developing
coronary heart disease within the next 10 years.

This notebook focuses on understanding the dataset,
identifying data-quality issues and exploring relationships
between different variables.

**Target variable:** TenYearCHD

**Machine-learning task:** Binary Classification

In [0]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [0]:

df = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/heart.csv"
)

df.head()

## 1. Initial Data Exploration

In this section, we examine the structure and
basic characteristics of the dataset.

In [0]:
# Check the number of rows and columns

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [0]:

# Display dataset information

df.info()

In [0]:
# Display summary statistics

df.describe().T

%md
## 3. Missing Value Analysis

Missing values are examined to identify incomplete records
and determine which attributes may require treatment during
the preprocessing stage.

In [0]:
# Count missing values in each column

missing_count = df.isnull().sum()

missing_count

In [0]:
# Calculate the percentage of missing values

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_percentage

In [0]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_summary

### Observation

There are missing values in several columns of the dataset.

The `diaBP` column has the highest number of missing values, with 412 missing records.

The target column `TenYearCHD` also has 361 missing values. Since this is the class label, these records cannot be used directly for training the model.

Other columns such as `diabetes`, `prevalentHyp`, `age`, `BMI`, `gender`, `heartRate`, `cigsPerDay`, and `totChol` also contain some missing values.

We will decide how to handle these missing values during the preprocessing stage.

%md
## 4. Duplicate Record Analysis

Duplicate records are checked to identify whether identical
patient records appear more than once in the dataset.

In [0]:
# Count duplicate rows

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

### Observation

No duplicate records were found in the dataset.

Therefore, there is no need to remove duplicate rows at this stage.

%md
## 5. Target Variable Analysis

The distribution of the target variable `TenYearCHD`
is examined to understand the number of records belonging
to each class and identify possible class imbalance.

In [0]:
# Count each class, including missing values

df["TenYearCHD"].value_counts(dropna=False)

In [0]:
df["TenYearCHD"].value_counts(normalize=True) * 100

### Observation

The target variable is imbalanced. Most records belong to
class 0, while a much smaller proportion belongs to class 1.



In [0]:
# Numerical variables

numerical_columns = [
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose"
]

# Categorical variables

categorical_columns = [
    "gender",
    "education",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes"
]

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

In [0]:
df[numerical_columns].hist(
    figsize=(14, 10),
    bins=30
)

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()

%md
### Observation

The numerical features show different distribution patterns.

- `age` is spread across the range from around 32 to 70 years.
- `cigsPerDay` has many values at 0 because many individuals are non-smokers. The distribution is strongly right-skewed.
- `totChol` is mostly concentrated around the middle values, but there are some very high values, creating a right-skewed distribution.
- `sysBP` is also right-skewed, with some high blood-pressure values.
- `diaBP` has a more balanced distribution, although some higher values can be seen.
- `BMI` is slightly right-skewed, with a few high values.
- `heartRate` is mostly concentrated around the middle range, with a few higher values.
- `glucose` is strongly right-skewed and contains some very high values.

These distributions suggest that some numerical features may contain outliers or extreme values. They will be examined further using boxplots before deciding whether any treatment is required.

In [0]:
for column in numerical_columns:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[column])
    plt.title(f"Boxplot of {column}")
    plt.show()

%md
### Observation

The boxplots show that several numerical features contain possible outliers.

- `age` does not show any clear outliers.
- `cigsPerDay` contains a few high values.
- `totChol` contains several high outliers and a few low values.
- `sysBP` contains many high outliers.
- `diaBP` contains both low and high outliers.
- `BMI` contains several high outliers.
- `heartRate` contains a few unusually low and high values.
- `glucose` contains many high outliers.

These values should not be removed immediately because some of them may be valid medical measurements. We will investigate them further during the preprocessing stage before deciding how to handle them.

%md
## 7. Categorical Feature Analysis

In this section, the categorical variables are examined to understand
the number of records in each category and to identify any unusual or
rare categories.

In [0]:
categorical_columns = [
    "gender",
    "education",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes"
]

In [0]:
for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

In [0]:
for column in categorical_columns:
    plt.figure(figsize=(6, 4))
    
    sns.countplot(
        data=df,
        x=column
    )
    
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.show()

%md
### Observation

The categorical features show different distributions in the dataset.

- There are more female records than male records.
- `currentSmoker` is almost evenly distributed between smokers and non-smokers.
- Most patients are not using blood-pressure medication (`BPMeds`).
- Only a very small number of patients have a previous stroke (`prevalentStroke`).
- More patients do not have hypertension, although a noticeable number of patients are diagnosed with hypertension.
- Most patients do not have diabetes, while only a small number are diabetic.
- For `education`, level 1 has the highest number of records, followed by levels 2, 3, and 4.

Some categorical features are highly uneven between their categories. These differences should be considered when analysing their relationship with the target variable.

In [0]:
# Create a temporary dataframe with known target values
# The original dataframe is not changed

df_target = df.dropna(subset=["TenYearCHD"]).copy()

In [0]:
for column in categorical_columns:
    plt.figure(figsize=(6, 4))

    sns.countplot(
        data=df_target,
        x=column,
        hue="TenYearCHD"
    )

    plt.title(f"{column} vs TenYearCHD")
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.legend(title="TenYearCHD")
    plt.show()

In [0]:
for column in categorical_columns:
    print(f"\nCHD percentage by {column}")

    percentage_table = pd.crosstab(
        df_target[column],
        df_target["TenYearCHD"],
        normalize="index"
    ) * 100

    percentage_table = percentage_table.round(2).reset_index()

    display(percentage_table)

### Observation

The percentage-based analysis gives a clearer comparison between the categorical features and `TenYearCHD`.

- Male patients have a higher percentage of CHD-positive cases than female patients.
- Current smokers show a slightly higher CHD-positive percentage than non-smokers.
- Patients using blood-pressure medication show a higher percentage of CHD-positive cases.
- Patients with a previous stroke show a higher CHD-positive percentage.
- Patients with hypertension have a noticeably higher percentage of CHD-positive cases than patients without hypertension.
- Diabetic patients also show a higher percentage of CHD-positive cases.
- Education level 1 has the highest CHD-positive percentage compared with the other education levels.

These results suggest that some categorical features may be useful for predicting the target variable. However, some categories contain only a small number of records, so the results should be interpreted carefully.

%md
## 8. Numerical Features vs Target Variable

In this section, numerical features are compared with the target variable `TenYearCHD` to identify whether there are differences between patients who developed CHD and those who did not.

In [0]:
for column in numerical_columns:
    plt.figure(figsize=(6, 4))

    sns.boxplot(
        data=df_target,
        x="TenYearCHD",
        y=column
    )

    plt.title(f"{column} vs TenYearCHD")
    plt.xlabel("TenYearCHD")
    plt.ylabel(column)
    plt.show()

In [0]:
group_summary = df_target.groupby("TenYearCHD")[numerical_columns].mean()

display(group_summary.round(2))

In [0]:
group_median = df_target.groupby("TenYearCHD")[numerical_columns].median()

display(group_median.round(2))

%md
### Observation

The numerical features show some differences between patients who developed CHD (`TenYearCHD = 1`) and those who did not (`TenYearCHD = 0`).

- Patients who developed CHD are generally older. Their average age is about 54 years, compared with about 49 years for the non-CHD group.
- `sysBP` is noticeably higher for the CHD-positive group, showing that systolic blood pressure may be an important feature.
- `diaBP` is also slightly higher for the CHD-positive group.
- `totChol` shows a small increase for the CHD-positive group.
- `BMI` is slightly higher among patients who developed CHD.
- `heartRate` is very similar between the two groups, so it may have a weaker relationship with the target.
- The average `glucose` level is higher for the CHD-positive group, although the median is the same for both groups. This may be due to some high glucose values affecting the mean.
- `cigsPerDay` has a higher average for the CHD-positive group, but the median is the same for both groups because many patients have low or zero cigarette consumption.

Overall, age and blood-pressure-related features show the clearest differences between the two target classes.

%md
## 9. Correlation Analysis

Correlation analysis is used to examine the relationship between numerical features.

It also helps identify features that may be strongly related to the target variable and features that may contain similar information.

In [0]:
# Select numerical features together with the target

correlation_columns = numerical_columns + ["TenYearCHD"]

# Calculate correlation matrix
correlation_matrix = df_target[correlation_columns].corr()

correlation_matrix

In [0]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")
plt.show()

In [0]:
target_correlation = correlation_matrix["TenYearCHD"].sort_values(
    ascending=False
)

target_correlation

%md
### Observation

The correlation analysis shows that some numerical features have a positive relationship with the target variable `TenYearCHD`.

- `age` has the highest positive correlation with `TenYearCHD` at approximately 0.23.
- `sysBP` also shows a positive correlation with the target at approximately 0.20.
- `diaBP` and `glucose` show smaller positive correlations with the target.
- `BMI`, `totChol`, `cigsPerDay`, and `heartRate` show relatively weak linear relationships with `TenYearCHD`.

A strong correlation of approximately 0.76 can be seen between `sysBP` and `diaBP`. This means that these two blood-pressure features contain similar information.

This possible redundancy will be considered during the preprocessing and feature-engineering stage.

Overall, no single numerical feature has a very strong linear correlation with the target, which suggests that the final prediction may depend on a combination of several features rather than one feature alone.

%md
## 10. Potential Data Leakage

Data leakage happens when information that should not be available during model training is accidentally used, causing the model to perform unrealistically well.

For this project, we will take several steps to reduce the risk of data leakage:

- The target variable `TenYearCHD` will not be used as an input feature.
- Rows with missing target values will be handled separately before model training.
- The dataset will be split into training and testing sets before fitting preprocessing steps such as imputation and scaling.
- Any resampling technique used to handle class imbalance will only be applied to the training data.
- Feature engineering and feature selection decisions will be made carefully so that information from the test set is not used during training.

These steps will help ensure that the final model is evaluated fairly on unseen data.

%md
## 11. Final EDA Summary

From the exploratory data analysis, several important findings were identified.

- The dataset contains 11,000 records and 16 columns.
- Missing values are present in several features, and the target variable `TenYearCHD` also contains missing values.
- No duplicate records were found.
- The target variable is imbalanced, with many more class `0` records than class `1` records.
- Several numerical features contain possible outliers, especially `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate`, and `glucose`.
- Some numerical variables, such as `cigsPerDay`, `sysBP`, `BMI`, and `glucose`, show skewed distributions.
- Patients with CHD tend to be older and generally have higher blood-pressure values.
- Some categorical features such as hypertension, diabetes, blood-pressure medication use, and previous stroke show higher CHD-positive percentages.
- `age` and `sysBP` have the highest positive correlations with `TenYearCHD` among the numerical features.
- `sysBP` and `diaBP` have a strong correlation with each other, showing possible feature redundancy.

These findings will guide the preprocessing and feature-engineering steps in the next stage.